In [3]:
import pandas as pd
import re

In [ ]:
base_path_data = "../../../data"
base_path_cv = base_path_data + "/processed/model_performance/cross_validation"
base_path_held_out = base_path_data + "/processed/model_performance/held_out"

# read metadata
metadata = pd.read_json(f"{base_path_data}/raw/metadata_ku.json", lines=True)
metadata_unique = metadata.groupby("new_sub_id").first().reset_index()

## Cross validation model performance

In [5]:
# function to extract subject ID from evaluation results filename
def extract_sub_id(filename):
    match = re.match(r"colon_(0*\d+)-", filename)
    if match:
        return f"sub{int(match.group(1)):03d}"
    return None

# read results
cv_results = {}
for fold_number in range(5):
    df = pd.read_csv(f"{base_path_cv}/results_fold_{fold_number}.csv")
    # group by filename and keep only the first row per file as we look at overall performance
    df = df.groupby("filename").first().reset_index()
    df["sub_id"] = df["filename"].apply(extract_sub_id)

    # merge evaluation results with metadata on subject ID (per scan)
    df = pd.merge(
        df,
        metadata_unique,
        left_on="sub_id",
        right_on="new_sub_id",
        how="left"
    )

    df["gender"] = df["gender"].apply(lambda x: x if x in ["M", "F"] else "U")
    grouped_by_gender =  df.groupby("gender")

    metrics = {
        "dice": {
            "count": grouped_by_gender["overall_dice_score"].count(),
            "mean": grouped_by_gender["overall_dice_score"].mean(),
            "std": grouped_by_gender["overall_dice_score"].std()
        },
        "hf95": {
            "count": grouped_by_gender["overall_hausdorff_distance_95th"].count(),
            "mean": grouped_by_gender["overall_hausdorff_distance_95th"].mean(),
            "std": grouped_by_gender["overall_hausdorff_distance_95th"].std()
        },
        "assd": {
            "count": grouped_by_gender["overall_average_symmetric_surface_distance"].count(),
            "mean": grouped_by_gender["overall_average_symmetric_surface_distance"].mean(),
            "std": grouped_by_gender["overall_average_symmetric_surface_distance"].std()
        },
    }
        
    cv_results[fold_number] = metrics

In [6]:
# print results
for fold in cv_results:
    print(f"------------------------ FOLD {fold} ------------------------")
    print(f"-------------- DICE --------------")
    print(pd.DataFrame(cv_results[fold]["dice"]))
    print(f"-------------- HF95 --------------")
    print(pd.DataFrame(cv_results[fold]["hf95"]))
    print(f"-------------- ASSD --------------")
    print(pd.DataFrame(cv_results[fold]["assd"]))


------------------------ FOLD 0 ------------------------
-------------- DICE --------------
        count      mean      std
gender                          
F          70  0.971808  0.01442
M          55  0.969084  0.02650
U          12  0.964214  0.02210
-------------- HF95 --------------
        count      mean       std
gender                           
F          70  2.424610  4.443671
M          55  2.596959  5.355975
U          12  4.256145  5.507208
-------------- ASSD --------------
        count      mean       std
gender                           
F          70  0.535022  0.362869
M          55  0.554227  0.408648
U          12  0.778512  0.550994
------------------------ FOLD 1 ------------------------
-------------- DICE --------------
        count      mean       std
gender                           
F          66  0.973294  0.011489
M          56  0.970655  0.013785
U          15  0.963584  0.026107
-------------- HF95 --------------
        count      mean        std
g

In [8]:
df.sort_values(by="overall_hausdorff_distance_95th", ascending=False).head(20)

,filename_x,component_id,hausdorff_distance_95th,overlap_percentage,component_size,overall_hausdorff_distance_95th,average_symmetric_surface_distance,overall_average_symmetric_surface_distance,dice_score,overall_dice_score,...,sub_id_y,patients_age,slice_location,gender,scan,position,mha_path,dicom_path,split,segmentation_path
57,colon_0258-supine.mha,1,1.000000,99.313184,1057488,52.939529,0.329426,2.670479,0.973557,0.954373,...,1.3.6.1.4.1.9328.50.4.0370,50.0,-551.8,M,1,supine,converted/sub258/sub258_pos-supine_scan-1_conv...,raw/sub258/sub258_pos-supine_scan-1.zip,NaN,NaN
27,colon_0113-supine.mha,1,39.268307,80.516326,3100288,39.255573,3.003467,3.003590,0.890479,0.890470,...,1.3.6.1.4.1.9328.50.4.0519,60.0,-424.975006,F,1,supine,converted/sub113/sub113_pos-supine_scan-1_conv...,raw/sub113/sub113_pos-supine_scan-1.zip,train,segmentations/segmentations-regionalgrowing-qc...
7,colon_0055-prone.mha,1,349.061768,99.778388,111456,34.380226,220.743881,2.411726,0.107589,0.944877,...,1.3.6.1.4.1.9328.50.4.0090,64.0,-519.015015,M,1,supine,converted/sub055/sub055_pos-supine_scan-1_conv...,raw/sub055/sub055_pos-supine_scan-1.zip,NaN,NaN
45,colon_0187-prone.mha,1,1.000000,99.660244,1329779,26.702061,0.351388,1.671560,0.976724,0.953946,...,1.3.6.1.4.1.9328.50.4.0447,60.0,-488.450012,F,1,supine,converted/sub187/sub187_pos-supine_scan-1_conv...,raw/sub187/sub187_pos-supine_scan-1.zip,NaN,NaN
46,colon_0187-supine.mha,1,1.000000,99.063305,955274,24.020824,0.305901,1.430327,0.974603,0.942900,...,1.3.6.1.4.1.9328.50.4.0447,60.0,-488.450012,F,1,supine,converted/sub187/sub187_pos-supine_scan-1_conv...,raw/sub187/sub187_pos-supine_scan-1.zip,NaN,NaN
50,colon_0218-supine.mha,1,1.000000,96.401123,2045527,20.904545,0.426274,1.433208,0.980200,0.961563,...,1.3.6.1.4.1.9328.50.4.0420,50.0,-488.910004,F,1,prone,converted/sub218/sub218_pos-prone_scan-1_conv-...,raw/sub218/sub218_pos-prone_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...
60,colon_0278-prone.mha,1,195.320114,99.247376,330444,11.000000,62.345264,1.307171,0.641072,0.928135,...,1.3.6.1.4.1.9328.50.4.0361,50.0,-465.420013,F,1,supine,converted/sub278/sub278_pos-supine_scan-1_conv...,raw/sub278/sub278_pos-supine_scan-1.zip,NaN,NaN
51,colon_0222-supine.mha,1,136.908890,91.433168,545721,8.000000,42.773674,1.101642,0.400750,0.940975,...,1.3.6.1.4.1.9328.50.4.0424,50.0,-679.1,F,1,prone,converted/sub222/sub222_pos-prone_scan-1_conv-...,raw/sub222/sub222_pos-prone_scan-1.zip,NaN,NaN
136,colon_0825-prone.mha,1,7.071068,94.137283,2652661,7.071068,0.924146,1.476229,0.968052,0.963999,...,1.3.6.1.4.1.9328.50.4.0786,59.0,-504.015015,M,1,prone,converted/sub825/sub825_pos-prone_scan-1_conv-...,raw/sub825/sub825_pos-prone_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...
102,colon_0465-prone.mha,1,5.656854,94.994804,2265366,5.656854,0.788211,0.788460,0.972910,0.972902,...,1.3.6.1.4.1.9328.50.4.0087,56.0,-508.325012,M,1,prone,converted/sub465/sub465_pos-prone_scan-1_conv-...,raw/sub465/sub465_pos-prone_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...


## Held out test set model performance

In [93]:
# function to extract subject ID from evaluation results filename
def extract_sub_id(filename):
    match = re.match(r"colon_(0*\d+)-", filename)
    if match:
        return f"sub{int(match.group(1)):03d}"
    return None

# read results
held_out_results = {}
file_paths = ["results_held_out_collapsed.csv", "results_held_out_noncollapsed.csv", "combined"]

for file_path in file_paths:
    if file_path == "combined":
        df1 = pd.read_csv(f"{base_path_held_out}/{file_paths[0]}")
        df2 = pd.read_csv(f"{base_path_held_out}/{file_paths[1]}")
        df = pd.concat([df1, df2])
    else:
        df = pd.read_csv(f"{base_path_held_out}/{file_path}")

    # group by filename and keep only the first row per file as we look at overall performance
    df = df.groupby("filename").first().reset_index()
    df["sub_id"] = df["filename"].apply(extract_sub_id)

    # merge evaluation results with metadata on subject ID (per scan)
    df = pd.merge(
        df,
        metadata_unique,
        left_on="sub_id",
        right_on="new_sub_id",
        how="left"
    )

    df["gender"] = df["gender"].apply(lambda x: x if x in ["M", "F"] else "U")
    grouped_by_gender =  df.groupby("gender")

    metrics = {
        "dice": {
            "count": grouped_by_gender["overall_dice_score"].count(),
            "mean": grouped_by_gender["overall_dice_score"].mean(),
            "std": grouped_by_gender["overall_dice_score"].std()
        },
        "hf95": {
            "count": grouped_by_gender["overall_hausdorff_distance_95th"].count(),
            "mean": grouped_by_gender["overall_hausdorff_distance_95th"].mean(),
            "std": grouped_by_gender["overall_hausdorff_distance_95th"].std()
        },
        "assd": {
            "count": grouped_by_gender["overall_average_symmetric_surface_distance"].count(),
            "mean": grouped_by_gender["overall_average_symmetric_surface_distance"].mean(),
            "std": grouped_by_gender["overall_average_symmetric_surface_distance"].std()
        },
    }
        
    held_out_results[file_path] = metrics

In [98]:
for results in held_out_results:
    print(f"------------------------ {results} ------------------------")
    print(f"-------------- DICE --------------")
    print(pd.DataFrame(held_out_results[results]["dice"]))
    print(f"-------------- HF95 --------------")
    print(pd.DataFrame(held_out_results[results]["hf95"]))
    print(f"-------------- ASSD --------------")
    print(pd.DataFrame(held_out_results[results]["assd"]))


------------------------ results_held_out_collapsed.csv ------------------------
-------------- DICE --------------
        count      mean       std
gender                           
F          26  0.972054  0.008802
M          12  0.974527  0.004285
-------------- HF95 --------------
        count      mean       std
gender                           
F          26  1.207219  0.494295
M          12  1.069036  0.161232
-------------- ASSD --------------
        count      mean       std
gender                           
F          26  0.452789  0.170509
M          12  0.418653  0.057460
------------------------ results_held_out_noncollapsed.csv ------------------------
-------------- DICE --------------
        count      mean       std
gender                           
F          14  0.970012  0.010681
M          16  0.970741  0.009986
U           2  0.965536  0.001763
-------------- HF95 --------------
        count      mean       std
gender                           
F          14 